# SOME TITLE
---

## Introduction

- provide some relevant background information on the topic so that someone unfamiliar with it will be prepared to understand the rest of your report
- clearly state the question you tried to answer with your project
- identify and fully describe the dataset that was used to answer the question

## Methods and results

- You may include references if necessary, as long as they all have a consistent citation style.

- describe the methods you used to perform your analysis from beginning to end that narrates the analysis code.

your report should include code which:
- loads data 
- wrangles and cleans the data to the format necessary for the planned analysis
- performs a summary of the data set that is relevant for exploratory data analysis related to the planned analysis 
- creates a visualization of the dataset that is relevant for exploratory data analysis related to the planned analysis
- performs the data analysis
- creates a visualization of the analysis 
- note: all figures should have a figure number and a legend


In [18]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor

import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor  # Good for regression
from sklearn.linear_model import Ridge  # Alternative linear model
from sklearn.pipeline import make_pipeline

In [3]:
import pandas as pd
players = pd.read_csv("https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz")
players

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN
...,...,...,...,...,...,...,...,...,...
191,Amateur,True,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,0.0,Bailey,Female,17,NaN,NaN
192,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,0.3,Pascal,Male,22,NaN,NaN
193,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,0.0,Dylan,Prefer not to say,17,NaN,NaN
194,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,2.3,Harlow,Male,17,NaN,NaN


In [5]:
'''
1) import, data cleansing, and merge

2) Correlation analysis for variables in players

3) Split-testing

4) K-variables

5) Clustering
'''

'\n1) import, data cleansing, and merge\n\n2) Correlation analysis for variables in players\n\n3) Split-testing\n\n4) K-variables\n\n5) Clustering\n'

In [13]:
### Run this cell before continuing.
import numpy as np
import pandas as pd
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn import set_config


# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

# Merge through shared hashedEmail
players = pd.read_csv("players.csv")
sessions = pd.read_csv("sessions.csv")
players_merged = players.merge(sessions, on="hashedEmail", how="inner")
players_clean = players_merged.drop(columns=["individualId", "organizationName"])

# Calculate each playtime duration in minutes
players_clean['start_time'] = pd.to_datetime(players_clean['start_time'], format='%d/%m/%Y %H:%M')
players_clean['end_time'] = pd.to_datetime(players_clean['end_time'], format='%d/%m/%Y %H:%M')
players_clean['duration_minutes'] = (players_clean['end_time'] - players_clean['start_time']).dt.total_seconds() / 60

# Calculate total playtime
agg = (
    players_clean.groupby("hashedEmail").agg(
        total_sessions=("hashedEmail", "count"),
        avg_session_length_minutes=("duration_minutes", "mean"),
        total_session_time_minutes=("duration_minutes", "sum")
    ).reset_index()
)

# Merge agg with players
players_total = players_clean.merge(agg, on="hashedEmail", how="inner")

# Get rid of duplicates and unneccessary data
players_total = players_total.drop(columns=["original_start_time", "original_end_time", "start_time", "end_time", "duration_minutes", "played_hours"])
players_total = players_total.drop_duplicates()



# Check to see if we can do clustering (Nope)
columns_to_plot = players_total.loc[:, "age" : "total_session_time_minutes"].columns.tolist()

pm_pairs = alt.Chart(players_total).mark_circle(opacity=0.2).encode(
    alt.X(alt.repeat("row"), type="quantitative"),
    alt.Y(alt.repeat("column"), type="quantitative"),
).properties(
    width=150,
    height=150
).repeat(
    column=columns_to_plot,
    row=columns_to_plot
)
pm_pairs

# Add new categorical column based on a threshold

# use 75th percentile as threshold
threshold = players_total["total_session_time_minutes"].quantile(0.75)

players_total["high_data_player"] = (
    players_total["total_session_time_minutes"] >= threshold
)

players_total



,experience,subscribe,hashedEmail,name,gender,age,total_sessions,avg_session_length_minutes,total_session_time_minutes,high_data_player
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,Morgan,Male,9,27,74.777778,2019.0,True
27,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,Christian,Male,17,3,85.000000,255.0,True
30,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,Blake,Male,17,1,5.000000,5.0,False
31,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,Flora,Female,21,1,50.000000,50.0,False
32,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,Kylie,Male,21,1,9.000000,9.0,False
...,...,...,...,...,...,...,...,...,...,...
1525,Veteran,True,ba24bebe588a34ac546f8559850c65bc90cd9d51b82158...,Gabriela,Female,44,1,11.000000,11.0,False
1526,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,Pascal,Male,22,1,21.000000,21.0,False
1527,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,Dylan,Prefer not to say,17,1,5.000000,5.0,False
1528,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,Harlow,Male,17,6,29.833333,179.0,True


In [14]:
tp_train, tp_test = train_test_split(players_total, test_size = 0.25, random_state = 123)
tp_train

,experience,subscribe,hashedEmail,name,gender,age,total_sessions,avg_session_length_minutes,total_session_time_minutes,high_data_player
93,Veteran,True,5a340c0e3d1aa3e579efc625bd3e5bca7fc25f7115b68e...,Zoe,Male,20,2,16.5,33.0,False
39,Amateur,True,3caa832978e0596779f4ee7c686c4592fb6de893925025...,Thatcher,Male,22,1,12.0,12.0,False
1412,Regular,True,d43af3bed5e9f1f31077233697c18f3f988a217bd0376a...,Xia,Female,20,1,32.0,32.0,False
846,Veteran,True,e44041459da2102dc20147ed6f0db4753547be66fc4dde...,Gianna,Male,18,1,26.0,26.0,False
243,Regular,True,f2826fb8dbce4d450348f99cb27ade184b713998d96797...,Zane,Male,10,7,38.0,266.0,True
...,...,...,...,...,...,...,...,...,...,...
1413,Regular,True,c121e4d197469bea90e21c0495001f4e21824adb98cbc6...,Rupert,Male,21,1,9.0,9.0,False
1262,Regular,True,7d71c49cbbce8dcf0276b2bfecfa2d16f22cb31a402455...,Devin,Two-Spirited,99,1,8.0,8.0,False
1255,Amateur,True,2cfed571797b66cc810c32562fc5b0f70b5bec0f525079...,Milo,Male,16,1,6.0,6.0,False
830,Beginner,True,96e190b0bf3923cd8d349eee467c09d1130af143335779...,Ibrahim,Prefer not to say,27,8,21.0,168.0,True


In [21]:
from sklearn.preprocessing import OneHotEncoder

pt_preprocessor = make_column_transformer(
    (StandardScaler(), ["total_sessions", "avg_session_length_minutes"]),  # Scale numerics only
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), ["experience", "subscribe"]),  # Encode categoricals
    remainder="drop"  # Drop target column and other non-features
)

# Now rebuild pipeline
pt_pipeline = make_pipeline(
    pt_preprocessor,
    RandomForestRegressor(random_state=123, n_jobs=-1)
)


In [22]:
pt_preprocessor = make_column_transformer(
                        (StandardScaler(), ["total_sessions", "avg_session_length_minutes", "total_session_time_minutes"]),
                        remainder = "passthrough",
                        verbose_feature_names_out = False
)

pt_preprocessor.fit(tp_train)
pt_scaled = pt_preprocessor.transform(tp_train)
pt_scaled.head()

,total_sessions,avg_session_length_minutes,total_session_time_minutes,experience,subscribe,hashedEmail,name,gender,age,high_data_player
93,-0.267863,-0.464968,-0.270330,Veteran,True,5a340c0e3d1aa3e579efc625bd3e5bca7fc25f7115b68e...,Zoe,Male,20,False
39,-0.289246,-0.617260,-0.278250,Amateur,True,3caa832978e0596779f4ee7c686c4592fb6de893925025...,Thatcher,Male,22,False
1412,-0.289246,0.059596,-0.270708,Regular,True,d43af3bed5e9f1f31077233697c18f3f988a217bd0376a...,Xia,Female,20,False
846,-0.289246,-0.143461,-0.272970,Veteran,True,e44041459da2102dc20147ed6f0db4753547be66fc4dde...,Gianna,Male,18,False
243,-0.160947,0.262653,-0.182459,Regular,True,f2826fb8dbce4d450348f99cb27ade184b713998d96797...,Zane,Male,10,True


In [26]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline

# Features and binary target
feature_cols = ['total_sessions', 'avg_session_length_minutes', 'experience', 'subscribe']
X = tp_train[feature_cols]
y = tp_train['high_data_player']

# Preprocessor (scaling critical for KNN distance calculations)
pt_preprocessor = make_column_transformer(
    (StandardScaler(), ["total_sessions", "avg_session_length_minutes"]),
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), ["experience", "subscribe"]),
    remainder="drop"
)


pt_pipeline = make_pipeline(
    pt_preprocessor,
    KNeighborsClassifier(n_neighbors=5, metric='euclidean')  # k=5 common choice
)

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=123)

accuracy_scores = cross_val_score(pt_pipeline, X, y, cv=kf, scoring='accuracy')
f1_scores = cross_val_score(pt_pipeline, X, y, cv=kf, scoring='f1')

print("KNN Accuracy scores:", accuracy_scores)
print(f"Mean Accuracy: {accuracy_scores.mean():.3f} (+/- {accuracy_scores.std() * 2:.3f})")
print("F1 scores:", f1_scores)
print(f"Mean F1: {f1_scores.mean():.3f} (+/- {f1_scores.std() * 2:.3f})")


KNN Accuracy scores: [0.68421053 0.84210526 0.89473684 0.94444444 0.88888889]
Mean Accuracy: 0.851 (+/- 0.179)
F1 scores: [0.5 0.  0.8 0.8 0.5]
Mean F1: 0.520 (+/- 0.585)


In [27]:
# Test different k values
k_values = range(1, 11)
cv_scores = []

for k in k_values:
    knn = make_pipeline(
        pt_preprocessor,
        KNeighborsClassifier(n_neighbors=k)
    )
    scores = cross_val_score(knn, X, y, cv=5, scoring='f1')
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f"Best k: {best_k} (F1: {max(cv_scores):.3f})")


Best k: 5 (F1: 0.652)


In [28]:
# Use best k for final model
pt_pipeline = make_pipeline(
    pt_preprocessor,
    KNeighborsClassifier(n_neighbors=best_k)
)

pt_pipeline.fit(X, y)
X_test = tp_test[feature_cols]
y_test = tp_test['high_data_player']

test_accuracy = pt_pipeline.score(X_test, y_test)
print(f"Test Accuracy with k={best_k}: {test_accuracy:.3f}")


Test Accuracy with k=5: 0.781


In [60]:
### Run this cell before continuing.
import numpy as np
import pandas as pd
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn import set_config


# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

# Merge through shared hashedEmail
players = pd.read_csv("players.csv")
sessions = pd.read_csv("sessions.csv")
players_merged = players.merge(sessions, on="hashedEmail", how="inner")
players_clean = players_merged.drop(columns=["individualId", "organizationName"])

# Calculate each playtime duration in minutes
players_clean['start_time'] = pd.to_datetime(players_clean['start_time'], format='%d/%m/%Y %H:%M')
players_clean['end_time'] = pd.to_datetime(players_clean['end_time'], format='%d/%m/%Y %H:%M')
players_clean['duration_minutes'] = (players_clean['end_time'] - players_clean['start_time']).dt.total_seconds() / 60

# Calculate total playtime
agg = (
    players_clean.groupby("hashedEmail").agg(
        total_sessions=("hashedEmail", "count"),
        avg_session_length_minutes=("duration_minutes", "mean"),
        total_session_time_minutes=("duration_minutes", "sum")
    ).reset_index()
)

# Merge agg with players
players_total = players_clean.merge(agg, on="hashedEmail", how="inner")

# Get rid of duplicates and unneccessary data
players_total = players_total.drop(columns=["original_start_time", "original_end_time", "start_time", "end_time", "duration_minutes", "played_hours"])
players_total = players_total.drop_duplicates()


players_total

columns_to_plot = players_total.loc[:, "age" : "total_session_time_minutes"].columns.tolist()


pm_pairs = alt.Chart(players_total).mark_circle(opacity=0.2).encode(
    alt.X(alt.repeat("row"), type="quantitative"),
    alt.Y(alt.repeat("column"), type="quantitative"),
).properties(
    width=150,
    height=150
).repeat(
    column=columns_to_plot,
    row=columns_to_plot
)
pm_pairs



alt.RepeatChart(...)

## Discussion

- summarize what you found
- discuss whether this is what you expected to find?
- discuss what impact could such findings have?
- discuss what future questions could this lead to?

## References